# Cargar_datos

## Contexto

En un proceso productivo este notebook no existiría: los datos de entrenamiento deberían estar
materializados en una tabla del DWH o Datalake de la empresa, como resultado de un proceso de
ingesta independiente (ETL/ELT) que se encarga de extraer, versionar y publicar la información.
El pipeline del modelo solo consumiría esa tabla.

Para este proyecto se usa un dataset **no productivo de ejemplo** (`BD_creditos.xlsx`). Este
notebook simula el paso de materialización: lee el archivo fuente, hace unas comprobaciones
mínimas de integridad y deja la información en un archivo plano `Base_de_datos.csv`, que es la
"tabla" que consumirán las etapas siguientes (EDA, ingeniería de características, entrenamiento).

En este paso **no se transforma** ningún dato: la limpieza y las correcciones se documentan en
`Comprension_eda.ipynb` y se implementarán en la etapa de ingeniería de características.

In [1]:
from pathlib import Path

import pandas as pd

# Rutas relativas a la raíz del repositorio (donde se ejecuta el notebook).
ARCHIVO_FUENTE = Path("BD_creditos.xlsx")
HOJA_FUENTE = "Hoja1"
ARCHIVO_SALIDA = Path("Base_de_datos.csv")

### Lectura del archivo fuente

Se valida que el archivo exista antes de leerlo para que el error sea explícito si cambia la
ubicación. Se fija la hoja `Hoja1` para no depender del orden de las hojas del libro.

In [2]:
if not ARCHIVO_FUENTE.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente: {ARCHIVO_FUENTE.resolve()}")

df = pd.read_excel(ARCHIVO_FUENTE, sheet_name=HOJA_FUENTE, engine="openpyxl")
print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Datos cargados: 10763 filas y 23 columnas


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


### Comprobaciones mínimas de integridad

Son controles de "contrato" entre la fuente y el resto del pipeline, no de calidad de datos
(esos se definen en el EDA):

- que estén todas las columnas esperadas, para detectar cambios de esquema en la fuente;
- que el archivo no venga vacío;
- que la variable objetivo `Pago_atiempo` exista y no tenga nulos.

In [3]:
COLUMNAS_ESPERADAS = [
    "tipo_credito", "fecha_prestamo", "capital_prestado", "plazo_meses", "edad_cliente",
    "tipo_laboral", "salario_cliente", "total_otros_prestamos", "cuota_pactada", "puntaje",
    "puntaje_datacredito", "cant_creditosvigentes", "huella_consulta", "saldo_mora",
    "saldo_total", "saldo_principal", "saldo_mora_codeudor", "creditos_sectorFinanciero",
    "creditos_sectorCooperativo", "creditos_sectorReal", "promedio_ingresos_datacredito",
    "tendencia_ingresos", "Pago_atiempo",
]

faltantes = set(COLUMNAS_ESPERADAS) - set(df.columns)
adicionales = set(df.columns) - set(COLUMNAS_ESPERADAS)
assert not faltantes, f"Faltan columnas en la fuente: {faltantes}"
assert len(df) > 0, "El archivo fuente está vacío"
assert df["Pago_atiempo"].notna().all(), "La variable objetivo tiene nulos"

if adicionales:
    print(f"Aviso: la fuente trae columnas no esperadas: {adicionales}")
print("Esquema validado: 23 columnas esperadas presentes, objetivo sin nulos.")
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")

Esquema validado: 23 columnas esperadas presentes, objetivo sin nulos.
Filas duplicadas exactas: 0


### Materialización

Se guarda sin el índice de pandas para que el CSV tenga exactamente las columnas de la fuente.
Las fechas quedan como texto ISO (`AAAA-MM-DD hh:mm:ss`) y se convierten a tipo fecha en el EDA.

In [4]:
df.to_csv(ARCHIVO_SALIDA, index=False)
print(f"Archivo generado: {ARCHIVO_SALIDA} ({ARCHIVO_SALIDA.stat().st_size / 1024:.0f} KB)")

Archivo generado: Base_de_datos.csv (1434 KB)
